# Test Saved Yield Model

This notebook loads `../../models/yield_model.pkl` and predicts crop yield from user-provided inputs. It stays separate from disease prediction.

## 1. Imports and Load Model

In [1]:
import os

import joblib
import pandas as pd
from IPython.display import display

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None
    print("ipywidgets is unavailable. Use the manual input cell at the bottom.")

DATA_PATH = "../../yield_data/yield_df.csv"
MODEL_PATH = "../../models/yield_model.pkl"

df = pd.read_csv(DATA_PATH).drop(columns=["Unnamed: 0"], errors="ignore").dropna()
model = joblib.load(MODEL_PATH)

areas = sorted(df["Area"].unique())
crops = sorted(df["Item"].unique())

print("Yield model loaded successfully.")
print(f"Areas: {len(areas)}")
print(f"Crops: {len(crops)}")

Yield model loaded successfully.
Areas: 101
Crops: 10


## 2. Predict With Form

In [2]:
def predict_yield(area, crop, year, rainfall, pesticides, avg_temp):
    row = pd.DataFrame([{
        "Area": area,
        "Item": crop,
        "Year": int(year),
        "average_rain_fall_mm_per_year": float(rainfall),
        "pesticides_tonnes": float(pesticides),
        "avg_temp": float(avg_temp),
    }])
    prediction = model.predict(row)[0]
    return row, prediction

if widgets is None:
    print("ipywidgets unavailable. Use the manual input cell below.")
else:
    area_widget = widgets.Dropdown(options=areas, value=areas[0], description="Area")
    crop_widget = widgets.Dropdown(options=crops, value=crops[0], description="Crop")
    year_widget = widgets.IntText(value=2024, description="Year")
    rainfall_widget = widgets.FloatText(value=float(df["average_rain_fall_mm_per_year"].median()), description="Rainfall")
    pesticide_widget = widgets.FloatText(value=float(df["pesticides_tonnes"].median()), description="Pesticides")
    temp_widget = widgets.FloatText(value=float(df["avg_temp"].median()), description="Avg temp")
    button = widgets.Button(description="Predict Yield", button_style="success")
    output = widgets.Output()

    def on_click(_):
        with output:
            output.clear_output()
            row, prediction = predict_yield(
                area_widget.value, crop_widget.value, year_widget.value,
                rainfall_widget.value, pesticide_widget.value, temp_widget.value
            )
            display(row)
            print(f"Predicted yield: {prediction:.2f} hg/ha")
            print(f"Predicted yield: {prediction / 10000:.4f} tonnes/ha")

    button.on_click(on_click)
    display(widgets.VBox([area_widget, crop_widget, year_widget, rainfall_widget, pesticide_widget, temp_widget, button, output]))

## Optional: Manual Input Fallback

In [3]:
manual_input = {
    "Area": "India",
    "Item": "Maize",
    "Year": 2024,
    "average_rain_fall_mm_per_year": 1083.0,
    "pesticides_tonnes": 50000.0,
    "avg_temp": 25.0,
}

row = pd.DataFrame([manual_input])
prediction = model.predict(row)[0]
display(row)
print(f"Predicted yield: {prediction:.2f} hg/ha")
print(f"Predicted yield: {prediction / 10000:.4f} tonnes/ha")

,Area,Item,Year,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,India,Maize,2024,1083.0,50000.0,25.0


Predicted yield: 25705.85 hg/ha
Predicted yield: 2.5706 tonnes/ha
